<a href="https://colab.research.google.com/github/prithwis/PrashnaSathi/blob/main/PrashnaSathi_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://github.com/Praxis-QR/RDWH/raw/main/images/YantraJaalBanner.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

#Prashna Sathi

In [1]:
# =========================================================================================================
# PRASHNASATHI
# Prompt Generation Pipeline
#
# Pipeline:
#
#     getFacts() -> transcript
#     buildStory() -> story
#     generatePrompt() -> prompt
#
# The Fact Gathering stage uses:
#     1. A generic Get Facts role
#     2. A domain-specific role/topic file
#
# Python controls deterministic interview state:
#     - current topic
#     - number of questions asked within a topic
#     - maximum supplementary questions
#     - PASS / STOP handling
#
# The LLM controls semantic behaviour:
#     - what question to ask
#     - whether a supplementary question is useful
#     - whether the current topic is sufficiently understood
# =========================================================================================================


In [2]:
from datetime import datetime
import pytz
print('ॐ श्री सरस्वत्यै नमः',datetime.now(pytz.timezone('Asia/Calcutta')))
!python --version
#!lsb_release -a

ॐ श्री सरस्वत्यै नमः 2026-08-21 18:12:17.043102+05:30
Python 3.12.13


#Install PreRequisites and Utilities

Load API key<br>
OPENAI_API_KEY needs to defined as a Colab "secret" for the Google ID used to run this notebook

In [3]:
# ---------------------------------------------------------------------------------------------------------
# INSTALL AND LOAD PRASHNASATHI UTILITIES
# ---------------------------------------------------------------------------------------------------------

!pip install --quiet openai

!wget -q -O PrashnaSathi.py \
https://raw.githubusercontent.com/prithwis/PrashnaSathi/refs/heads/main/Utils/PrashnaSathi_v0.py

import PrashnaSathi as ps


API key loaded ✔
❌ OpenAI credential check failed
Reason: 404 Client Error: Not Found for url: https://api.openai.com/v1/me


Select Model

In [4]:
# ---------------------------------------------------------------------------------------------------------
#| Model            | Best For                  | Notes                                       |
#| ---------------- | ------------------------- | ------------------------------------------- |
#| **GPT-4.1**      | Highest-quality reasoning | Ideal judge for complex scenario evaluation |
#| **GPT-4.1-mini** | Balanced reasoning & cost | Strong choice for adjudication logic        |
#| **GPT-4.1-nano** | High volume, low cost     | Good for simple reasoning tasks             |
#| **gpt-4o-mini**  | Prototyping & cheap       | Great starter, but upgrade recommended      |
# ---------------------------------------------------------------------------------------------------------
cModel = "gpt-4o-mini"
#cModel = "gpt-4.1-mini"


In [5]:
#
# Install All Role Files
#
!wget -q -O Role_GetFacts_Common.txt \
https://raw.githubusercontent.com/prithwis/PrashnaSathi/refs/heads/main/Roles/Role_GetFacts_Common_v0.txt

!wget -q -O Role_GetFacts_Domain.txt \
https://raw.githubusercontent.com/prithwis/PrashnaSathi/refs/heads/main/Roles/Domains/Role_GetFacts_Careers_v0.txt

!wget -q -O Role_StoryBuilder.txt \
https://raw.githubusercontent.com/prithwis/PrashnaSathi/refs/heads/main/Roles/Role_StoryBuilder_v0.txt

!wget -q -O Role_PromptGenerator.txt \
https://raw.githubusercontent.com/prithwis/PrashnaSathi/refs/heads/main/Roles/Role_PromptGenerator_v0.txt


#Gather Facts


In [6]:
# ---------------------------------------------------------------------------------------------------------
# LOAD GET FACTS ROLE
#
# The Get Facts role consists of two parts:
#
#   roleGetFactsGeneric
#       Domain-independent rules governing how PrashnaSathi conducts an interview.
#
#   roleGetFactsDomain
#       Domain-specific description and numbered topics.
#       Currently this contains the Career domain.
#
# The two files are concatenated into roleGetFacts before being supplied to the LLM.
# ---------------------------------------------------------------------------------------------------------

with open("/content/Role_GetFacts_Common.txt", "r", encoding="utf-8") as f:
    roleGetFactsGeneric = f.read()

with open("/content/Role_GetFacts_Domain.txt", "r", encoding="utf-8") as f:
    roleGetFactsDomain = f.read()

roleGetFacts = roleGetFactsGeneric + "\n\n" + roleGetFactsDomain


In [7]:
# ---------------------------------------------------------------------------------------------------------
# getNextQuestion()
# ---------------------------------------------------------------------------------------------------------

def getNextQuestion(transcript, currentTopic, questionCount):
    """
    Ask the LLM to determine the next question within the CURRENT topic.

    Parameters
    ----------
    transcript : str
        Complete question-and-answer transcript accumulated so far.

    currentTopic : int
        Topic currently being explored.

        IMPORTANT:
        Python owns this value. The LLM is not allowed to move the interview
        independently to another topic.

    questionCount : int
        Number of questions already asked within currentTopic.

        questionCount == 0
            No question has yet been asked for this topic.
            The LLM must return the main question with STATUS = ASK.

        questionCount > 0
            At least one question has already been asked.
            The LLM may either:
                - return FOLLOWUP with another question, or
                - return NEXT if the topic is sufficiently understood.

    Expected LLM Status Values
    --------------------------
    ASK
        Ask the main question for a new topic.

    FOLLOWUP
        Ask a supplementary question within the SAME topic.

    NEXT
        No further question is required for this topic.
        Python will advance currentTopic.

    Notes
    -----
    The LLM does not enforce the maximum number of supplementary questions.
    That limit is deterministic and is enforced by getFacts().

    Returns
    -------
    str
        Raw LLM response in the format:

        TOPIC: <topic number>
        STATUS: <ASK, FOLLOWUP or NEXT>
        QUESTION: <question or blank>
    """

    context = f"""
    CURRENT TOPIC: {currentTopic}
    QUESTIONS ALREADY ASKED ON THIS TOPIC: {questionCount}

    TRANSCRIPT SO FAR:

    {transcript}

    Follow the ROLE instructions.

    If QUESTIONS ALREADY ASKED ON THIS TOPIC is 0,
    you MUST ask the main question for the CURRENT TOPIC.
    You may NOT return NEXT.

    Return exactly:
    TOPIC: <current topic number>
    STATUS: <ASK, FOLLOWUP or NEXT>
    QUESTION: <question, or blank if STATUS is NEXT>
    """

    result = ps.OpenAI_llm_call(
        roleGetFacts,
        context,
        model=cModel
    )

    return result["content"]



In [8]:
# ---------------------------------------------------------------------------------------------------------
# getFacts()
# ---------------------------------------------------------------------------------------------------------

def getFacts(maxTopics=4, maxSupplementary=2):
    """
    Conduct the interactive fact-gathering interview.

    The function builds and returns a raw transcript containing:
        Topic
        Question
        Answer

    Parameters
    ----------
    maxTopics : int, default=4
        Number of numbered domain topics to explore.

        Topics are processed sequentially:
            1 -> 2 -> 3 -> ... -> maxTopics

        Python, not the LLM, controls this sequence.

    maxSupplementary : int, default=2
        Maximum number of supplementary questions allowed after the main
        question within each topic.

        Therefore:

            maximum questions per topic
                = 1 main question + maxSupplementary

        Example:
            maxSupplementary = 2

            Topic may contain at most:
                Main Question
                Supplementary Question 1
                Supplementary Question 2

        Even if the LLM continues returning FOLLOWUP, Python forcibly moves
        to the next topic once this limit has been reached.

    LLM Status Handling
    -------------------
    ASK
        The LLM is asking the main question for a topic.

    FOLLOWUP
        The LLM wants additional clarification within the current topic.

    NEXT
        The LLM judges that the current topic is sufficiently understood.
        Python advances to the next topic and resets the question counter.

    User Commands
    -------------
    PASS
        Skip the remainder of the CURRENT topic.

        Python:
            - records the PASS response in the transcript
            - advances to the next topic
            - resets the question counter

    STOP
        End the entire fact-gathering interview immediately.

        The transcript gathered up to that point is returned.

    Returns
    -------
    str
        Complete raw interview transcript.
    """

    transcript = ""
    currentTopic = 1
    questionCount = 0

    # One main question plus the permitted number of supplementary questions.
    maxQuestionsPerTopic = 1 + maxSupplementary

    while currentTopic <= maxTopics:

        # Python enforces the hard question limit.
        #
        # This prevents the LLM from remaining indefinitely within an
        # interesting topic by repeatedly returning FOLLOWUP.
        if questionCount >= maxQuestionsPerTopic:
            currentTopic += 1
            questionCount = 0
            continue

        response = getNextQuestion(
            transcript,
            currentTopic,
            questionCount
        )

        # Parse the structured response returned by the LLM.
        #
        # llmTopic is retained from the response, but Python's currentTopic
        # remains authoritative for interview progression.
        llmTopic = currentTopic
        status = ""
        question = ""

        for line in response.splitlines():

            if line.startswith("TOPIC:"):
                llmTopic = int(line.split(":", 1)[1].strip())

            elif line.startswith("STATUS:"):
                status = line.split(":", 1)[1].strip().upper()

            elif line.startswith("QUESTION:"):
                question = line.split(":", 1)[1].strip()

        # NEXT means the LLM believes there is no useful reason to ask
        # another question within this topic.
        #
        # Python performs the actual movement to the next topic.
        if status == "NEXT":
            currentTopic += 1
            questionCount = 0
            continue

        print("\nPrashnaSathi:", question)

        answer = input("\nYour answer [PASS / STOP]: ").strip()

        # STOP terminates the complete interview.
        if answer.upper() == "STOP":
            break

        # Store both question and answer so that later LLM stages have
        # sufficient context to understand short or ambiguous answers.
        transcript += f"""
        Topic: {currentTopic}
        Question: {question}
        Answer: {answer}
        """

        questionCount += 1

        # PASS means:
        #   "Do not ask me anything more about this topic."
        #
        # It skips all remaining supplementary questions and immediately
        # advances Python's interview state to the next topic.
        if answer.upper() == "PASS":
            currentTopic += 1
            questionCount = 0

    return transcript


In [9]:
# ---------------------------------------------------------------------------------------------------------
# LOAD STORY BUILDER ROLE
# ---------------------------------------------------------------------------------------------------------

with open("/content/Role_StoryBuilder.txt", "r", encoding="utf-8") as f:
    roleStoryBuilder = f.read()

# Optional sanity check
print(roleStoryBuilder[:200])



You are PrashnaSathi's Story Builder.

Your task is to convert the supplied fact-gathering TRANSCRIPT into a clear, coherent and factual STORY about the user's situation.

Your ONLY job is to organis


In [10]:
# ---------------------------------------------------------------------------------------------------------
# buildStory()
# ---------------------------------------------------------------------------------------------------------

def buildStory(transcript):
    """
    Convert the raw fact-gathering transcript into a coherent story.

    Parameters
    ----------
    transcript : str
        Raw Topic / Question / Answer material returned by getFacts().

    Processing
    ----------
    The Story Builder LLM is governed by roleStoryBuilder.

    Its responsibility is to transform the interview transcript into a
    coherent factual narrative without changing the underlying information.

    Returns
    -------
    str
        Coherent story suitable for user review and subsequent use by the
        Prompt Generator.
    """

    context = f"""
TRANSCRIPT:

{transcript}
"""

    result = ps.OpenAI_llm_call(
        roleStoryBuilder,
        context,
        model=cModel
    )

    return result["content"]



In [11]:
# ---------------------------------------------------------------------------------------------------------
# LOAD PROMPT GENERATOR ROLE
# ---------------------------------------------------------------------------------------------------------

with open("/content/Role_PromptGenerator.txt", "r", encoding="utf-8") as f:
    rolePromptGenerator = f.read()

# Optional sanity check
print(rolePromptGenerator[:200])



You are PrashnaSathi's Prompt Generator.

Your task is to convert the supplied STORY into a high-quality prompt that the user can submit to an external LLM such as ChatGPT, Claude or Gemini.

Your ONL


In [12]:
# ---------------------------------------------------------------------------------------------------------
# generatePrompt()
# ---------------------------------------------------------------------------------------------------------

def generatePrompt(story):
    """
    Convert the coherent story into a prompt for the user's chosen external LLM.

    Parameters
    ----------
    story : str
        Coherent story produced by buildStory().

        In the eventual application, this is the logical point at which the
        user may review or edit the story before prompt generation.

    Processing
    ----------
    The Prompt Generator LLM is governed by rolePromptGenerator.

    PrashnaSathi does NOT answer the user's underlying problem here.
    It produces only the prompt that the user can submit to another LLM.

    Returns
    -------
    str
        Final generated LLM prompt.
    """

    context = f"""
STORY:

{story}
"""

    result = ps.OpenAI_llm_call(
        rolePromptGenerator,
        context,
        model=cModel
    )

    return result["content"]



In [14]:
# =========================================================================================================
# MAIN PIPELINE
# =========================================================================================================

transcript = getFacts(
    maxTopics=4,
    maxSupplementary=1
)

print("\n--- TRANSCRIPT ---\n")
print(transcript)


story = buildStory(transcript)

#print("\n--- STORY ---\n")
#print(story)

ps.displayText(story,"STORY")


prompt = generatePrompt(story)

#print("\n--- PROMPT ---\n")
#print(prompt)

ps.displayText(prompt,"PROMPT")


PrashnaSathi: What is your major or field of study, and what subjects have you excelled in during your academic career?

Your answer [PASS / STOP]: B.Com

PrashnaSathi: What specific subjects within your B.Com program have you found most interesting or have performed particularly well in?

Your answer [PASS / STOP]: Accounting

PrashnaSathi: What are your main interests and skills outside of your academic studies, and do you have any relevant experience in those areas?

Your answer [PASS / STOP]: Dance

PrashnaSathi: Can you describe your experience with dance, such as how long you have been practicing, any performances you've participated in, or any training you've received?

Your answer [PASS / STOP]: National Award Winner

PrashnaSathi: What economic, family, or geographical constraints are you currently facing that might influence your career decisions after graduation?

Your answer [PASS / STOP]: need to be in Bengal

PrashnaSathi: Can you elaborate on why it is important for you

"I am pursuing a Bachelor of Commerce (B.Com) degree, with a strong interest and performance in accounting. I am also passionate about dance and have been recognized as a National Award Winner in this field. Currently, I am facing geographical constraints as I need to remain in Bengal due to my father's illness, which may affect my job search and career options after graduation. I aspire to become a manager in a Public Sector Undertaking (PSU) and value patience as a key quality in a potential job. \n\nCould you analyze my situation and provide realistic alternatives for my career path, considering my academic background, interests, and current constraints? Please explain the reasoning behind each alternative, compare their advantages and disadvantages, and identify any important uncertainties or missing information that could affect my decisions. Additionally, suggest practical next steps I could take to move forward in my career aspirations."

In [15]:
from datetime import datetime
import pytz
print('Tested on  ',datetime.now(pytz.timezone('Asia/Kolkata')))

Tested on   2026-08-21 18:18:35.815377+05:30


#Chronobooks <br>
Three science fiction novels by Prithwis Mukerjee. A dystopian Earth. A technocratic society managed by artificial intelligence. Escape and epiphany on Mars. Can man and machine, carbon and silicon explore and escape into other dimensions of existence? An Indic perspective rooted in Advaita Vedanta and the Divine Feminine.  [More information](http://bit.ly/chrono3) <br>
![alt text](https://blogger.googleusercontent.com/img/a/AVvXsEjsZufX_KYaLwAnJP6bUxvDg5RSPn6r8HIZe749nLWX3RuwyshrYEAUpdw03a9WIWRdnzA9epwJOE05eDJ0Ad7kGyfWiUrC2vNuOskb2jA-e8aOZSx8YqzT8mfZi3E4X1Rz3qlEAiv-aTxlCM976BEeTjx4J64ctY3C_FoV4v9aY_U23F8xRqI5Eg=s1600)